# ORBITGUARD Stage 1: Data collection and verification

This notebook downloads the original OPSSAT-AD files from Zenodo, checks their published MD5 checksums, and inspects their schemas, labels, and official split. It does not perform exploratory analysis or train a model. Run the cells from top to bottom. The original CSV labels are never modified.

## 1. Prepare the notebook environment

The next cell imports the small set of tools needed to download files, verify checksums, and read CSVs. The path logic works when the notebook is launched from the project root or its `notebooks` folder, and it also creates the same `data/raw` folder when run in Google Colab.

In [6]:
import hashlib
import time
from pathlib import Path
from urllib.error import URLError
from urllib.request import urlopen

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
print(f"Project folder: {PROJECT_ROOT}")
print(f"Raw data folder: {RAW_DATA_DIR}")

Project folder: c:\Users\HP\OneDrive\Desktop\orbit gaurd
Raw data folder: c:\Users\HP\OneDrive\Desktop\orbit gaurd\data\raw


## 2. Download the original Zenodo files

The file names and MD5 values below come from the Zenodo record metadata. A file already on disk is reused only if its checksum matches. Downloads are retried on network errors and are written to a temporary file first; a checksum mismatch or repeated network failure stops with an explicit error.

In [7]:
ZENODO_FILE_URL = "https://zenodo.org/api/records/12588359/files/{filename}/content"
EXPECTED_MD5 = {
    "dataset.csv": "c12fb83863afa4463dec07303a20bb14",
    "segments.csv": "1f4b6ffd0359efac96a6698ff557e0f9",
}

def md5_file(path):
    """Calculate an MD5 checksum without loading the whole file into memory."""
    digest = hashlib.md5()
    with path.open("rb") as file_handle:
        for chunk in iter(lambda: file_handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def download_and_verify(filename, expected_md5, attempts=3):
    """Download a Zenodo file and keep it only after its checksum matches."""
    destination = RAW_DATA_DIR / filename
    if destination.exists() and md5_file(destination) == expected_md5:
        print(f"Already downloaded and verified: {filename}")
        return destination

    temporary_path = destination.with_name(destination.name + ".part")
    for attempt in range(1, attempts + 1):
        try:
            with urlopen(ZENODO_FILE_URL.format(filename=filename), timeout=90) as response:
                with temporary_path.open("wb") as output_file:
                    while True:
                        chunk = response.read(1024 * 1024)
                        if not chunk:
                            break
                        output_file.write(chunk)

            actual_md5 = md5_file(temporary_path)
            if actual_md5 != expected_md5:
                raise ValueError(
                    f"Checksum mismatch for {filename}: expected {expected_md5}, "
                    f"received {actual_md5}. The downloaded file was not accepted."
                )

            temporary_path.replace(destination)
            print(f"Downloaded and verified: {filename} ({destination.stat().st_size:,} bytes)")
            return destination
        except URLError as error:
            if temporary_path.exists():
                temporary_path.unlink()
            if attempt == attempts:
                raise RuntimeError(
                    f"Could not download {filename} after {attempts} attempts."
                ) from error
            time.sleep(2 ** (attempt - 1))
        except Exception:
            if temporary_path.exists():
                temporary_path.unlink()
            raise

downloaded_files = {
    filename: download_and_verify(filename, checksum)
    for filename, checksum in EXPECTED_MD5.items()
}
print("\nVerified files:")
for filename, path in downloaded_files.items():
    print(f"- {filename}: {path.stat().st_size:,} bytes; MD5 {md5_file(path)}")

Already downloaded and verified: dataset.csv
Already downloaded and verified: segments.csv

Verified files:
- dataset.csv: 507,568 bytes; MD5 c12fb83863afa4463dec07303a20bb14
- segments.csv: 17,970,946 bytes; MD5 1f4b6ffd0359efac96a6698ff557e0f9


## 3. Inspect every downloaded CSV

This cell reads each CSV as-is and prints its row/column count, every column name, inferred pandas data types, and three sample records. Long text values in the sample are shortened only for display; the files on disk are unchanged.

In [8]:
csv_paths = sorted(RAW_DATA_DIR.glob("*.csv"))
if {path.name for path in csv_paths} != set(EXPECTED_MD5):
    raise FileNotFoundError(
        "Expected dataset.csv and segments.csv in data/raw. "
        "Run the download cell before inspecting the data."
    )

dataframes = {}
for csv_path in csv_paths:
    frame = pd.read_csv(csv_path)
    dataframes[csv_path.name] = frame
    print(f"\n{'=' * 72}\n{csv_path.name}")
    print(f"Shape: {frame.shape[0]:,} rows × {frame.shape[1]} columns")
    print("Columns:", frame.columns.tolist())
    print("Data types:\n", frame.dtypes.to_string())
    sample = frame.head(3).copy()
    for column in sample.columns:
        sample[column] = sample[column].map(
            lambda value: value[:100] if isinstance(value, str) else value
        )
    print("Sample records:\n", sample.to_string(index=False))


dataset.csv
Shape: 2,123 rows × 23 columns
Columns: ['segment', 'anomaly', 'train', 'channel', 'sampling', 'duration', 'len', 'mean', 'var', 'std', 'kurtosis', 'skew', 'n_peaks', 'smooth10_n_peaks', 'smooth20_n_peaks', 'diff_peaks', 'diff2_peaks', 'diff_var', 'diff2_var', 'gaps_squared', 'len_weighted', 'var_div_duration', 'var_div_len']
Data types:
 segment               int64
anomaly               int64
train                 int64
channel                 str
sampling              int64
duration              int64
len                   int64
mean                float64
var                 float64
std                 float64
kurtosis            float64
skew                float64
n_peaks               int64
smooth10_n_peaks      int64
smooth20_n_peaks      int64
diff_peaks            int64
diff2_peaks           int64
diff_var            float64
diff2_var           float64
gaps_squared          int64
len_weighted          int64
var_div_duration    float64
var_div_len         float64
Sa

## 4. Confirm the target and published train/test split

The benchmark's `anomaly` column is the segment-level target. Its `train` column provides the official split (`1` for training and `0` for test). The cross-tabulation below reports the actual counts in the downloaded file. The raw sample-level `label` values are printed separately so categorical labels are not confused with the binary target.

In [9]:
feature_dataset = dataframes["dataset.csv"]
raw_segments = dataframes["segments.csv"]

required_columns = {"anomaly", "train"}
missing_columns = required_columns.difference(feature_dataset.columns)
if missing_columns:
    raise ValueError(f"The feature table is missing required columns: {sorted(missing_columns)}")

print("Segment-level target counts (anomaly):")
print(feature_dataset["anomaly"].value_counts(dropna=False).sort_index().to_string())
print("\nSegment-level official split counts (train):")
print(feature_dataset["train"].value_counts(dropna=False).sort_index().to_string())
print("\nOfficial split by target:")
print(pd.crosstab(feature_dataset["train"], feature_dataset["anomaly"]).to_string())
print("\nRaw telemetry label counts:")
print(raw_segments["label"].value_counts(dropna=False).to_string())
print("\nRaw label by binary anomaly target:")
print(pd.crosstab(raw_segments["label"], raw_segments["anomaly"]).to_string())

Segment-level target counts (anomaly):
anomaly
0    1689
1     434

Segment-level official split counts (train):
train
0     529
1    1594

Official split by target:
anomaly     0    1
train             
0         416  113
1        1273  321

Raw telemetry label counts:
label
a3         106930
anomaly    100264
a2          75272
a4          21027

Raw label by binary anomaly target:
anomaly       0       1
label                  
a2        75272       0
a3       106930       0
a4        21027       0
anomaly       0  100264


## 5. Separate benchmark features from identifiers and metadata

`dataset.csv` contains 18 handcrafted benchmark features along with fields that must not be passed to a model as features: `segment` is an identifier, `channel` and `sampling` are metadata, `anomaly` is the target, and `train` is the split indicator. The following code uses the feature allowlist from the project loader and checks that no excluded fields appear in it. This is a schema check only; it does not transform labels or train a model.

In [10]:
import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import ENGINEERED_FEATURE_COLUMNS, select_engineered_features

engineered_features = select_engineered_features(feature_dataset)
excluded_fields = {"segment", "channel", "sampling", "anomaly", "train"}
assert not excluded_fields.intersection(engineered_features.columns)
print(f"Verified engineered feature count: {engineered_features.shape[1]}")
print("Engineered feature columns:", list(ENGINEERED_FEATURE_COLUMNS))
print("Excluded identifier/metadata/target/split fields:", sorted(excluded_fields))

Verified engineered feature count: 18
Engineered feature columns: ['duration', 'len', 'mean', 'var', 'std', 'kurtosis', 'skew', 'n_peaks', 'smooth10_n_peaks', 'smooth20_n_peaks', 'diff_peaks', 'diff2_peaks', 'diff_var', 'diff2_var', 'gaps_squared', 'len_weighted', 'var_div_duration', 'var_div_len']
Excluded identifier/metadata/target/split fields: ['anomaly', 'channel', 'sampling', 'segment', 'train']


## Stage 1 complete

Both original files have been verified against the Zenodo checksums, and their schemas, examples, target labels, and supplied train/test split have been inspected. Keep the original data under `data/raw/`; it is excluded from Git. Stop here for Stage 1—EDA and model training belong to later stages.